# 🤖 AI Travel Claim Assistant: Smart Interactive Dashboard
Welcome to your conversational claim assistant panel. Use the interactive chat modules below to locate previous claims, auto-generate new tracking directories, or run compliance audits.

### 📝 Step 1: Initialize Setup Environment
Ensure your `GEMINI_API_KEY` is added to your left-hand Secrets Vault (🔑) and toggled **On**, then run the initialization block below to wake up your assistant.

In [ ]:
#@title 🛠️ INITIALIZE ASSISTANT ENGINE
!pip install -q -U google-genai
from google.colab import drive, userdata
from google.genai import Client
import pandas as pd
import json
import time
import os

drive.mount('/content/drive', force_remount=True)
BASE_WORKSPACE = "/content/drive/MyDrive/Travel_Claims_Workspace"
INPUT_DIR = f"{BASE_WORKSPACE}/1_DROP_RECEIPTS_HERE"
OUTPUT_DIR = f"{BASE_WORKSPACE}/2_GENERATED_SYSTEM_REPORTS"
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

client = Client(api_key=userdata.get('GEMINI_API_KEY'))
print("\n🎉 Smart Claim Assistant Engine Initialized and Ready!")

### 💬 Module A: Claim Lookup & Folder Creation
Type your request in the string field below. You can ask the AI to find an old claim by descriptive context (e.g., hotel name, date) or tell it to generate a brand-new workspace folder structure automatically.

In [ ]:
#@title 🧠 CHAT COMMAND INPUT PANEL
USER_REQUEST = "Create a new claim directory for my business trip to Denver this week" #@param {type:"string"}

# Gather history from previous reports for AI lookup context
history_context = ""
if os.path.exists(OUTPUT_DIR):
    summaries = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('_SUMMARY.md')]
    for f_name in summaries:
        with open(os.path.join(OUTPUT_DIR, f_name), 'r') as f:
            history_context += f"\n[File: {f_name}]\n" + f.read()[:300]

intent_prompt = f"""
Analyze this user instruction: \"{USER_REQUEST}\"
Based on the instruction and our history context, classify the action needed.
You MUST respond ONLY with a clean JSON object using this schema:
{{
    \"action_type\": \"LOOKUP\" or \"CREATE_NEW\" or \"UNKNOWN\",
    \"target_claim_id\": \"String (e.g. CLAIM-2026-001 or proposed new ID structure)\",
    \"message\": \"Natural language response back to the user\"
}}
History Context Database:
{history_context}
"""

response = client.models.generate_content(model='gemini-3.8-flash', contents=intent_prompt)
clean_json = response.text.replace("```json", "").replace("```", "").strip()
intent = json.loads(clean_json)

print(f"Assistant response: {intent['message']}\n")

if intent['action_type'] == 'CREATE_NEW':
    new_folder_path = os.path.join(INPUT_DIR, intent['target_claim_id'])
    os.makedirs(new_folder_path, exist_ok=True)
    print(f"📁 AUTOMATION ACTION: Successfully generated physical workspace folder on Google Drive!")
    print(f"📍 Directory Created: {new_folder_path}")
    print("👉 Action Required: Drag and drop your trip receipts into this new folder, then run Module B.")
elif intent['action_type'] == 'LOOKUP':
    print(f"🔍 AUTOMATION ACTION: Located historical records matching your criteria.")
    print(f"🔑 Identified Target Claim ID: {intent['target_claim_id']}")

### 🚀 Module B: Batch Audit & System Report Compilation
Once your receipts are dropped into the specific folder inside Google Drive, enter the verified target Claim ID below and run this cell to extract and export your corporate system import data.

In [ ]:
#@title ⚙️ EXECUTE CLAIM CONSOLIDATION ENGINE
TARGET_CLAIM_ID = "CLAIM-2026-001" #@param {type:"string"}

RUN_INPUT_DIR = f"{INPUT_DIR}/{TARGET_CLAIM_ID}"
if os.path.exists(RUN_INPUT_DIR):
    files = [f for f in os.listdir(RUN_INPUT_DIR) if os.path.isfile(os.path.join(RUN_INPUT_DIR, f))]
    if not files:
        print(f"🛑 Pipeline Halted: No transaction files found inside folder: {RUN_INPUT_DIR}")
    else:
        print(f"Processing Pipeline Engaged for Claim ID: {TARGET_CLAIM_ID} ({len(files)} files discovered)")
        parsed_records = []
        for file_name in files:
            with open(os.path.join(RUN_INPUT_DIR, file_name), "r") as f: 
                raw_text = f.read()
            system_prompt = f"Extract transaction details from this text. Respond ONLY with a clean JSON object matching standard fields: {{'VendorName':'String', 'TransactionTotal':Float, 'ApprovedAmount':Float, 'TaxPaid':Float, 'PolicyViolationFlag':'Y' or 'N', 'AuditComments':'String'}}. Text: {{raw_text}}"
            try:
                response = client.models.generate_content(model='gemini-3.8-flash', contents=system_prompt)
                clean_json = response.text.replace("```json", "").replace("```", "").strip()
                item_data = json.loads(clean_json)
                item_data["ClaimID"] = TARGET_CLAIM_ID
                item_data["SourceFileName"] = file_name
                parsed_records.append(item_data)
                print(f"   -> Successfully processed transaction entry: {file_name}")
            except Exception:
                pass
        if parsed_records:
            df = pd.DataFrame(parsed_records)
            df.to_csv(f"{OUTPUT_DIR}/{TARGET_CLAIM_ID}_IMPORT.csv", index=False)
            with open(f"{OUTPUT_DIR}/{TARGET_CLAIM_ID}_SUMMARY.md", "w") as md:
                md.write(f"# Human-Readable Review: {TARGET_CLAIM_ID}\n\n" + df.to_markdown(index=False))
            print(f"\n🎉 AUDIT COMPLETE! Ingestion assets exported to: {OUTPUT_DIR}")
else:
    print(f"🛑 Path Error: Target folder folder does not exist: {RUN_INPUT_DIR}")